In [1]:
!git config --global user.name "Brendan Hills"
!git config --global user.email brendanhills@google.com

In [2]:
!python3 -m venv venv
!source venv/bin/activate
%pwd


from platform import python_version

print(python_version())

3.10.13


In [3]:
!pip3 -q install db-dtypes
!pip3 -q install "google-cloud-bigquery>=3.17"
!pip3 -q install "google-cloud-aiplatform>=1.38"
!pip3 -q install "pandas>=2.2.0"

In [ ]:
!gcloud auth application-default login

In [4]:

PROJECT_ID = "uk-bh-experiments-argolis"  # @param {type:"string"}
REGION = "US"  # @param {type: "string"}
DATASET_ID = "schema_mapping"  # @param {type:"string"}

In [5]:
import pandas as pd
from google.cloud import bigquery
from vertexai.language_models import TextEmbeddingModel
import numpy as np
from IPython.display import display, HTML

In [6]:
def get_table_sample(table_name):
  client = bigquery.Client()
  table_query = f"""
  SELECT * FROM {PROJECT_ID}.{DATASET_ID}.{table_name} TABLESAMPLE SYSTEM (10 PERCENT)
  """
  table_sample = client.query(table_query)
  table_sample_df = table_sample.to_dataframe()
  return table_sample_df




In [7]:
TABLENAME1="insurance"

table1_df = get_table_sample(TABLENAME1)

table1_df.head()



,age,sex,bmi,children,smoker,region,charges
0,18,female,26.315,0,False,northeast,2198.18985
1,18,female,38.665,2,False,northeast,3393.35635
2,18,female,35.625,0,False,northeast,2211.13075
3,18,female,30.115,0,False,northeast,21344.84670
4,18,male,23.750,0,False,northeast,1705.62450


In [8]:
TABLENAME2="df1_loan"

table2_df = get_table_sample(TABLENAME2)

table2_df.head()


,int64_field_0,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,Total_Income
0,63,LP001213,Male,True,1,Graduate,False,4945,0.0,NaN,360.0,0.0,Rural,False,4945.0
1,127,LP001449,Male,False,0,Graduate,False,3865,1640.0,NaN,360.0,1.0,Rural,True,5505.0
2,284,LP001922,Male,True,0,Graduate,False,20667,0.0,NaN,360.0,1.0,Rural,False,20667.0
3,322,LP002054,Male,True,2,Not Graduate,False,3601,1590.0,NaN,360.0,1.0,Rural,True,5191.0
4,231,LP001768,Male,True,0,Graduate,<NA>,3716,0.0,42.0,180.0,1.0,Rural,True,3716.0


In [13]:
def get_embedding_for_col(column):
  embeddings = []
  #print(f'{column=}')
  for row in column:
    embeddings.append(row)
  return embeddings

def get_embeddings_for_table(table):
  embeddings = []
  for col in table.columns:
    embeddings.append(get_embedding_for_col(table[col]))
  return embeddings


def text_embedding(text):
    """Text embedding with a Large Language Model."""
    model = TextEmbeddingModel.from_pretrained("textembedding-gecko")
    embeddings = model.get_embeddings(text)
    embedding_vector = []
    for embedding in embeddings:
        embedding_vector.append(embedding.values)
    return embedding_vector

In [14]:
def get_embedding_for_col2(column):
  print(f'{column=}')
  #convert column to list of strings
  col_strings = [str(cell) for cell in column]
  #convert list of strings to string
  col_string = ' '.join(col_strings)
  print(f'{col_string=}')
  embeddings = text_embedding([col_string])
  return embeddings

def get_embeddings_for_table2(table):
  embeddings = []
  for col in table.columns:
    embeddings.append(get_embedding_for_col2(table[col]))
  return embeddings




In [15]:
#embeddings_df1 = pd.DataFrame(get_embeddings_for_table2(table1_df))

#embeddings_df1.head()

In [16]:
## return a dataframe of embeddings - one column for each column of the table
def get_embeddings_for_table_columns(table):
    columns_strings = []
    #convert each column of the table into a list of strings
    for col in table.columns:
        #print(f'{col=}')
        #sort by that column so that the ordering of rows doesn't influence the embedding
        sorted_df = table.sort_values(col)
        #convert column to list of strings
        col_strings = [str(cell) for cell in sorted_df[col]]
        #convert list of strings to string
        col_string =  col + ' ' + ' '.join(col_strings)
        columns_strings.append(col_string)


    column_embeddings = text_embedding(columns_strings)
    embeddings_df = pd.DataFrame( index=table.columns, data=column_embeddings).transpose()
    return embeddings_df
    

In [17]:
embeddings_df1 = get_embeddings_for_table_columns(table1_df)

print ("Embeddings for Table 1:")
display(embeddings_df1)

Embeddings for Table 1:


,age,sex,bmi,children,smoker,region,charges
0,0.000102,-0.027570,-0.021312,0.006877,-0.010799,-0.032914,-0.003290
1,-0.024747,-0.033315,-0.039788,-0.001577,-0.053381,-0.009089,-0.022956
2,-0.040663,-0.049940,-0.041895,-0.059085,-0.052489,-0.059915,-0.047246
3,0.013269,0.036736,-0.000665,0.037384,-0.029618,-0.007890,-0.024515
4,0.070544,0.080333,0.071443,0.055929,0.090776,0.052379,0.058684
...,...,...,...,...,...,...,...
763,0.022872,0.000951,0.010719,0.007888,0.010809,-0.010496,0.009103
764,0.007379,0.024543,0.028144,0.030870,0.049990,0.003344,0.034328
765,0.039393,0.047356,0.080138,0.016164,0.058307,0.021966,0.013929
766,-0.036721,-0.022198,-0.053073,-0.049516,-0.006577,-0.026187,-0.068465


In [18]:
embeddings_df2 = get_embeddings_for_table_columns(table2_df)

print ("Embeddings for Table 2:")
display(embeddings_df2)

Embeddings for Table 2:


,int64_field_0,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,Total_Income
0,0.009032,-0.000066,-0.046166,0.005408,0.000965,-0.013101,-0.005680,-0.022290,0.001693,-0.017434,-0.006508,0.020411,-0.007956,-0.001873,0.006067
1,-0.007447,-0.031781,-0.046284,-0.044958,-0.029426,-0.070008,-0.038764,-0.060689,-0.049285,-0.030086,-0.034153,-0.050509,-0.079278,-0.037951,-0.037130
2,-0.052971,-0.028108,-0.044689,-0.046667,-0.047295,-0.020035,-0.017155,-0.069760,-0.060662,-0.037885,-0.045084,-0.032819,-0.047799,-0.063080,-0.055070
3,-0.003733,0.013256,0.032094,0.018725,0.001859,0.016171,0.008764,0.020583,0.033095,0.006064,0.015316,0.037934,0.030496,-0.005089,0.021040
4,0.049476,0.035676,0.091816,0.061410,0.087708,0.101595,0.109506,0.081576,0.091247,0.073410,0.063981,0.065743,0.064654,0.089227,0.071296
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
763,0.049515,-0.008123,0.024289,-0.001734,0.010698,0.018973,0.011161,0.003363,0.013673,0.014073,0.008138,0.011022,0.049482,0.022508,0.028774
764,0.017748,0.010998,0.029302,0.018760,0.044469,0.022605,0.023183,0.028852,0.029264,-0.000408,0.001631,0.026350,0.006020,0.006706,0.021188
765,0.019066,0.068847,0.034288,0.049742,0.035933,0.047628,0.058236,0.061074,0.057886,0.088691,0.090312,0.048667,0.034039,0.082431,0.054603
766,-0.054442,-0.056358,-0.025046,-0.029769,-0.069439,-0.035056,-0.017311,-0.036946,-0.049697,-0.049463,-0.057856,-0.051609,-0.041326,-0.036963,-0.038915


In [19]:
def vector_similarity(vec1, vec2):
    return np.dot(np.squeeze(np.array(vec1)),np.squeeze(np.array(vec2)))

In [20]:

#print (f'{distances_df=}')
rows_list = []
for row in embeddings_df1:
    row_distances = []
    for col in embeddings_df2:
        distance = vector_similarity(embeddings_df1[row], embeddings_df2[col])
        #print(f'{distance=}')
        row_distances.append(distance)    
    rows_list.append(row_distances)
        

distances_df = pd.DataFrame(index=embeddings_df1.columns,columns=embeddings_df2.columns, data=rows_list)
print ("Embeddings for Distances:")
distances_df.style \
    .background_gradient(cmap='Blues', axis=None) \
    .format(precision=2)
      


Embeddings for Distances:


,int64_field_0,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,Total_Income
age,0.80,0.76,0.79,0.76,0.70,0.80,0.74,0.81,0.76,0.79,0.80,0.68,0.74,0.71,0.76
sex,0.67,0.66,0.92,0.76,0.67,0.77,0.73,0.67,0.68,0.70,0.72,0.64,0.78,0.71,0.67
bmi,0.74,0.70,0.71,0.70,0.63,0.72,0.69,0.79,0.74,0.78,0.75,0.59,0.69,0.69,0.77
children,0.68,0.69,0.70,0.65,0.77,0.65,0.63,0.62,0.69,0.67,0.68,0.71,0.64,0.62,0.64
smoker,0.63,0.63,0.73,0.77,0.65,0.71,0.79,0.64,0.63,0.66,0.70,0.58,0.67,0.73,0.63
region,0.63,0.61,0.70,0.63,0.61,0.71,0.66,0.61,0.62,0.63,0.66,0.58,0.76,0.62,0.63
charges,0.74,0.74,0.65,0.64,0.65,0.67,0.65,0.81,0.76,0.77,0.73,0.63,0.63,0.66,0.82


Columns that should be similar:

 insurance.sex, df1_loan.gender
 insurance.children, df1_loan.Dependants